# DBSCAN Clustering on Well Log Data
## Machine learning for energy applications — Module 3, Exercise 2

In this exercise we apply **DBSCAN** (Density-Based Spatial Clustering of Applications
with Noise) to well log data. Unlike K-Means, DBSCAN does not require specifying the
number of clusters in advance and can discover clusters of arbitrary shape. It also labels
samples in sparse regions as **noise** (label −1). Noise means "not in a dense region at the
chosen `eps` and `min_samples`", not "bad data": noise points are candidates to inspect
(thin beds, fluid contacts, rare lithologies, tool effects) before anyone calls them anomalies.

---
## 0 — Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Load and Explore the Data

We load two datasets:
- **log.csv** — a two-feature dataset with SP (spontaneous potential) and GR (gamma ray)
- **w5.csv** — a multi-feature well log dataset

We will primarily work with `log.csv` for 2D visualization, and then apply DBSCAN
to the higher-dimensional `w5.csv`.

In [ ]:
# YOUR CODE HERE
# TODO: Load log.csv into a DataFrame called df_log
# TODO: Print the shape and column names
# TODO: Display the first 10 rows


In [ ]:
# YOUR CODE HERE
# TODO: Load w5.csv into a DataFrame called df_w5
# TODO: Print the shape and column names
# TODO: Display the first 10 rows


In [ ]:
# YOUR CODE HERE
# TODO: Display .describe() and .isnull().sum() for both DataFrames


In [ ]:
# YOUR CODE HERE
# TODO: Create a scatter plot of SP vs GR from df_log
# Hint: Use alpha=0.4 and s=15 for readability


---
## 2 — Standardize Features

DBSCAN relies on distance calculations, so features must be on a comparable scale.
We standardize both datasets.

> **Exploratory analysis of fixed datasets.** Each scaler is fitted on all samples of its
> dataset because we only describe these data: nothing is predicted for new samples, so there
> is no training/validation/test split (scikit-learn's DBSCAN cannot label new samples without
> refitting anyway). `eps` is measured in these standardized units.

In [ ]:
# YOUR CODE HERE
# TODO: Extract the SP and GR columns from df_log as a NumPy array (drop NaNs)
# TODO: Fit a StandardScaler on the log data and transform it into X_log_scaled
# TODO: Print the number of samples, scaled means, and scaled stds
#
# TODO: Define w5_features = ["Pressure", "AI", "VpVs", "SW", "phid", "vcl"]
# TODO: Extract and standardize the w5 data into X_w5_scaled
# TODO: Print the number of samples, scaled means, and scaled stds


---
## 3 — Apply DBSCAN with Default Parameters

DBSCAN has two key parameters:
- **eps** — the maximum distance between two samples for them to be considered neighbours
- **min_samples** — the minimum number of points required to form a dense region (core point)

We start with `eps=0.5` and `min_samples=5` on the 2D log data.

In [ ]:
# YOUR CODE HERE
# TODO: Create a DBSCAN model with eps=0.5 and min_samples=5
# TODO: Fit and predict labels on X_log_scaled
# TODO: Count the number of clusters (remember: label -1 = noise, not a cluster)
# TODO: Count and print the number of noise points
# TODO: Print the size of each cluster (including noise)
# Hint: n_clusters = len(set(labels)) - (1 if -1 in labels else 0)


---
## 4 — Visualize Clusters and Noise Points

One of DBSCAN's key advantages is its ability to label points as noise. We colour
cluster members and mark noise points separately.

In [ ]:
# YOUR CODE HERE
# TODO: Create a 1x2 subplot figure
# TODO: Left panel — scatter of original standardized data (single color)
# TODO: Right panel — scatter coloured by DBSCAN cluster label
#       Use black "x" markers for noise points (label == -1)
#       Use different colors for each cluster
# Hint: Loop through sorted(set(labels)) and use a different color per label


---
## 5 — K-Distance Plot to Estimate eps

The **K-distance plot** helps choose `eps`. For each point, we compute the distance to its
k-th nearest neighbour, with k = min_samples and the point itself counted as its first
neighbour (as DBSCAN's `min_samples` counts it). We sort these distances and look for the
"knee" where the curve starts to rise steeply: an `eps` near the knee keeps most samples
as core points and leaves the sparse tail as noise.

In [ ]:
# YOUR CODE HERE
# TODO: Use NearestNeighbors with n_neighbors=5 to find distances
# TODO: Extract the 5th nearest neighbour distance for each point
# TODO: Sort distances in ascending order
# TODO: Plot the sorted distances
# TODO: Add a horizontal line at the current eps=0.5 for reference
# TODO: Print distance statistics (min, median, 90th percentile, mean, max)
# TODO: Print the share of samples whose k-distance is <= 0.5
# Hint: nn.kneighbors() returns (distances, indices)
#       The k-th neighbour distance is distances[:, k-1]


---
## 6 — Experiment with Different eps and min_samples

We systematically vary `eps` and `min_samples` to observe how the clustering
results change. This helps build intuition for parameter tuning. The summary table reports
the number of clusters, the noise and the largest cluster; its silhouette is computed on the
clustered points only, because noise (−1) is not a cluster, so read it together with the
noise fraction.

In [ ]:
# YOUR CODE HERE
# TODO: Create a 2x3 subplot figure
# TODO: For eps_values = [0.2, 0.3, 0.5, 0.8, 1.0, 1.5] with fixed min_samples=5:
#       - Fit DBSCAN and plot the results
#       - Show number of clusters and noise points in the title
# Hint: Use black "x" markers for noise, coloured dots for clusters


In [ ]:
# YOUR CODE HERE
# TODO: Create a 2x3 subplot figure
# TODO: For min_samples_values = [3, 5, 10, 15, 20, 30] with fixed eps=0.5:
#       - Fit DBSCAN and plot the results
#       - Show number of clusters and noise points in the title


In [ ]:
# YOUR CODE HERE
# TODO: Create a summary table of parameter combinations
# TODO: For eps in [0.3, 0.5, 0.8, 1.0] and min_samples in [3, 5, 10, 15]:
#       record n_clusters, n_noise, noise_%, the largest cluster size, and the
#       silhouette score of the clustered points only (noise, label -1, is not a cluster)
# TODO: Display results as a DataFrame
# Hint: silhouette_score requires at least 2 clusters; use np.nan if fewer
#       clustered = labs != -1
#       silhouette_score(X_log_scaled[clustered], labs[clustered])


---
## 7 — Compare DBSCAN with K-Means on the Same Data

DBSCAN and K-Means make fundamentally different assumptions. K-Means assumes
compact, roughly spherical clusters of similar size. DBSCAN finds clusters based on density
and can detect arbitrarily shaped clusters and noise.

We keep `eps = 0.5`, `min_samples = 5`: 0.5 lies at the start of the bend in the k-distance
plot (Section 5), and the eps sweep in Section 6 (min_samples = 5) gives one cluster for every
eps from 0.5 to 1.5, so the conclusion does not hinge on the exact value. DBSCAN finds a single
cluster here, and K-Means needs at least two, so we fit K = 2 and discuss what each method is
saying. The DBSCAN silhouette is undefined for a single cluster.

In [ ]:
# YOUR CODE HERE
# TODO: Use the configuration supported by the k-distance plot and the sweep
#       (eps=0.5, min_samples=5)
# TODO: Fit DBSCAN and count the number of clusters found
# TODO: Fit K-Means with the same number of clusters (K-Means needs K >= 2:
#       if DBSCAN finds one cluster, use K = 2)
# TODO: Create a 1x2 subplot figure comparing both methods:
#       - Left: DBSCAN results with noise marked
#       - Right: K-Means results with centroids marked (use "*" marker)
# TODO: Print silhouette scores for both methods (exclude DBSCAN noise; the
#       DBSCAN silhouette is undefined for a single cluster)
# Hint: For K-Means centroids, use kmeans.cluster_centers_


---
## 8 — Apply DBSCAN to Multi-Feature Well Log Data (w5.csv)

Now we apply DBSCAN to the six standardized W5 logs. Distances grow with the number of
features, so we read `eps` again from the k-distance plot of these data instead of assuming
that the log.csv value transfers, and then check how the result changes with `eps`.

In [ ]:
# YOUR CODE HERE
# TODO: Create a K-distance plot for the w5 data to estimate eps
# TODO: Use NearestNeighbors with k=5
# TODO: Choose eps_w5 where the curve starts to rise steeply and mark it on the plot
# TODO: Print the median and 90th percentile of the k-distances, and the share of
#       samples whose k-distance is <= eps_w5
# Hint: read eps from this plot; do not assume the log.csv value transfers to six features


In [ ]:
# YOUR CODE HERE
# TODO: Apply DBSCAN to X_w5_scaled with your eps_w5 and min_samples=5
# TODO: Print the number of clusters, noise points and the largest cluster size
# TODO: Tabulate clusters, noise % and largest cluster for eps in [0.3, 0.5, 0.8, 1.0]
# TODO: Visualize the results in AI vs VpVs space
#       Mark noise points with black "x" markers


---
## 9 — Analyse Noise Points Detected by DBSCAN

Points labelled as noise (cluster = -1) lie in low-density regions at the chosen `eps` and
`min_samples`; that is not the same as bad data. Compare them with the other samples in
original units and check where they occur along depth before interpreting them: measurement
problems, thin beds, unusual lithologies and fluid contacts are all candidates.

In [ ]:
# YOUR CODE HERE
# TODO: Separate noise and normal points from the 2D log data DBSCAN results
# TODO: Print summary statistics in original units (use X_log) for noise points
#       and normal points separately
# Hint: Use labels == -1 to create a boolean mask for noise


In [ ]:
# YOUR CODE HERE
# TODO: Create a 1x2 subplot figure for outlier visualization:
#       - Left: scatter plot with noise highlighted in red, normal in blue
#       - Right: histogram comparing SP distributions of normal vs noise points
# Hint: Use density=True in hist() for comparable distributions


In [ ]:
# YOUR CODE HERE
# TODO: For the w5 data, compare noise vs other samples' feature means (original units)
# TODO: Create a comparison DataFrame showing Normal Mean, Outlier Mean, and Difference
# TODO: Print the share of samples with SW < 1 among noise and among the other samples
# TODO: Check whether noise points form thick intervals or thin runs along depth
#       (w5.csv is sorted by depth, so consecutive row positions are consecutive depths)
# Hint: Handle the case where no noise points are detected


---
## Summary

In this exercise we covered:

1. **Data Loading** — working with both 2D (log.csv) and multi-dimensional (w5.csv) well log data.
2. **Feature Standardization** — essential preprocessing for distance-based algorithms (exploratory: fitted on each fixed dataset).
3. **DBSCAN with Default Parameters** — applying DBSCAN and understanding its output (clusters + noise).
4. **Cluster Visualization** — plotting clusters and clearly marking noise points.
5. **K-Distance Plot** — reading `eps` from the knee of the sorted k-distances.
6. **Parameter Sensitivity** — tabulating clusters, noise and the largest cluster as eps and min_samples change.
7. **DBSCAN vs K-Means** — understanding the trade-offs between density-based and centroid-based clustering.
8. **Noise Analysis** — comparing noise points with the other samples, in original units and along depth, before calling them anomalies.

DBSCAN is particularly valuable in petrophysics when:
- The number of clusters is unknown
- Clusters have irregular shapes
- Low-density samples should be flagged for review (e.g., bad hole, thin beds, unusual formations)